# Walle GCS — Model Comparison (Results)

Draft results page for the AI models wired into the Walle GCS stack. Each section lists the model, where it is invoked in the repo, its size / accuracy / throughput characteristics, and a quick visual comparison.

**Status:** the metric tables below are **placeholders** — they use published model-card numbers (RF-DETR, Depth Anything V2, YOLOv8, InsightFace) and reasonable in-repo throughput estimates, *not* a benchmark run against this codebase. Replace them with measurements from your own runs (see the last section).

Models covered:

| Task | Model | Invoked in |
|---|---|---|
| Object detection | RF-DETR Base / Medium | `walle_gcs.py`, `walle_gcs_max.py` |
| Metric depth | Depth Anything V2 — Indoor-Small / Indoor-Base / Outdoor-Small / Outdoor-Base | `walle_gcs*.py`, `slam_engine.py` |
| Human pose + tracking | YOLOv8-pose + DeepSort (MobileNet ReID) + Kalman | `human_tracker_engine.py`, `core/` |
| Face | InsightFace `buffalo_l` (SCRFD + ArcFace) | `face_recognition_engine.py` |
| Visual odometry | OpenCV ORB + PnP keyframe VO | `slam_engine.py` (`KeyframeVO`) |
| 3D reconstruction | LingBot-Map (`lingbot-map-long.pt`) | `reconstruction_engine.py` |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 80)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Model inventory

All models the GCS can load at runtime, with the file that pulls them in and how the weights arrive on disk.

In [ ]:
inventory = pd.DataFrame([
    {"task": "detection",   "model": "RF-DETR Base",                                       "weights": "library default (HF)",                                              "loader": "walle_gcs.py: RFDETRBase()"},
    {"task": "detection",   "model": "RF-DETR Medium",                                     "weights": "library default (HF)",                                              "loader": "walle_gcs_max.py: RFDETRMedium()"},
    {"task": "depth",       "model": "Depth Anything V2 — Metric Indoor Small",             "weights": "depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf",            "loader": "walle_gcs.py / slam_engine.py"},
    {"task": "depth",       "model": "Depth Anything V2 — Metric Indoor Base",              "weights": "depth-anything/Depth-Anything-V2-Metric-Indoor-Base-hf",             "loader": "walle_gcs_max.py / slam_engine.py"},
    {"task": "depth",       "model": "Depth Anything V2 — Metric Outdoor Small",            "weights": "depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf",           "loader": "slam_engine.py (DEPTH_MODELS_SLAM)"},
    {"task": "depth",       "model": "Depth Anything V2 — Metric Outdoor Base",             "weights": "depth-anything/Depth-Anything-V2-Metric-Outdoor-Base-hf",            "loader": "slam_engine.py (DEPTH_MODELS_SLAM)"},
    {"task": "pose",        "model": "YOLOv8n-pose",                                       "weights": "yolov8n-pose.pt (repo root)",                                       "loader": "human_tracker_engine.py"},
    {"task": "tracking",    "model": "DeepSort + MobileNet ReID",                          "weights": "deep_sort_realtime default",                                        "loader": "human_tracker_engine.py"},
    {"task": "tracking",    "model": "Kalman tracker (constant-velocity)",                 "weights": "— (filterpy)",                                                       "loader": "core/kalman_tracker.py"},
    {"task": "face",        "model": "InsightFace SCRFD (detector)",                       "weights": "buffalo_l (ONNX, auto-download)",                                   "loader": "face_recognition_engine.py"},
    {"task": "face",        "model": "InsightFace ArcFace R100 (embedding)",               "weights": "buffalo_l (ONNX, auto-download)",                                   "loader": "face_recognition_engine.py"},
    {"task": "slam",        "model": "ORB keyframe VO + PnP",                              "weights": "— (OpenCV ORB)",                                                     "loader": "slam_engine.py: KeyframeVO"},
    {"task": "reconstruction", "model": "LingBot-Map",                                     "weights": "lingbot-map/checkpoints/lingbot-map-long.pt (external)",             "loader": "reconstruction_engine.py (subprocess)"},
])
inventory

## 2. Object detection — RF-DETR Base vs Medium

Both editions of the GCS run RF-DETR through `ObjectDetectionEngine` with `conf_thresh=0.5`. The MAX edition swaps `RFDETRBase()` for `RFDETRMedium()`. Numbers below come from the published RF-DETR card.

*Latency is a representative single-image RTX-class GPU number — not measured in this repo.*

In [ ]:
detection = pd.DataFrame([
    {"model": "RF-DETR Base",   "params_M": 29.0, "coco_mAP@0.5:0.95": 53.3, "gpu_latency_ms": 6.0, "input_res": 560},
    {"model": "RF-DETR Medium", "params_M": 38.0, "coco_mAP@0.5:0.95": 54.7, "gpu_latency_ms": 7.5, "input_res": 576},
]).set_index("model")
detection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
detection["params_M"].plot.bar(ax=axes[0], color=["#4c78a8", "#f58518"]); axes[0].set_title("Params (M)"); axes[0].set_xlabel("")
detection["coco_mAP@0.5:0.95"].plot.bar(ax=axes[1], color=["#4c78a8", "#f58518"]); axes[1].set_title("COCO mAP@0.5:0.95"); axes[1].set_xlabel(""); axes[1].set_ylim(50, 60)
detection["gpu_latency_ms"].plot.bar(ax=axes[2], color=["#4c78a8", "#f58518"]); axes[2].set_title("GPU latency / img (ms)"); axes[2].set_xlabel("")
plt.tight_layout(); plt.show()

## 3. Metric depth — Depth Anything V2 variants

Four checkpoints from the Depth Anything V2 family are referenced in the repo. `walle_gcs.py` ships Indoor-Small, `walle_gcs_max.py` ships Indoor-Base, and `slam_engine.py` exposes all four via `DEPTH_MODELS_SLAM`.

Lower AbsRel is better; higher δ<1.25 is better.

In [ ]:
depth = pd.DataFrame([
    {"variant": "Indoor-Small",  "backbone": "ViT-S", "params_M": 24.8, "benchmark": "NYUv2",   "AbsRel": 0.045, "delta_1.25": 0.979, "gpu_latency_ms": 12, "used_by": "walle_gcs.py, SLAM"},
    {"variant": "Indoor-Base",   "backbone": "ViT-B", "params_M": 97.5, "benchmark": "NYUv2",   "AbsRel": 0.043, "delta_1.25": 0.982, "gpu_latency_ms": 25, "used_by": "walle_gcs_max.py, SLAM"},
    {"variant": "Outdoor-Small", "backbone": "ViT-S", "params_M": 24.8, "benchmark": "KITTI",   "AbsRel": 0.080, "delta_1.25": 0.946, "gpu_latency_ms": 12, "used_by": "SLAM (outdoor preset)"},
    {"variant": "Outdoor-Base",  "backbone": "ViT-B", "params_M": 97.5, "benchmark": "KITTI",   "AbsRel": 0.075, "delta_1.25": 0.952, "gpu_latency_ms": 25, "used_by": "SLAM (outdoor preset)"},
]).set_index("variant")
depth

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
colors = ["#4c78a8", "#54a24b", "#f58518", "#e45756"]
depth["params_M"].plot.bar(ax=axes[0], color=colors); axes[0].set_title("Params (M)"); axes[0].set_xlabel("")
depth["AbsRel"].plot.bar(ax=axes[1], color=colors); axes[1].set_title("AbsRel (lower is better)"); axes[1].set_xlabel("")
depth["gpu_latency_ms"].plot.bar(ax=axes[2], color=colors); axes[2].set_title("GPU latency / frame (ms)"); axes[2].set_xlabel("")
for ax in axes:
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## 4. Human pose + tracking pipeline

`human_tracker_engine.py` chains three components. They are not interchangeable — the comparison here is *what each stage contributes*, not which one to pick.

In [ ]:
tracker = pd.DataFrame([
    {"stage": "detect/pose", "model": "YOLOv8n-pose",            "params_M": 3.3, "coco_kp_mAP": 49.7, "role": "per-frame person bboxes + 17 keypoints"},
    {"stage": "reid/assoc",  "model": "DeepSort + MobileNet",    "params_M": 3.5, "coco_kp_mAP": None, "role": "track ID continuity across frames"},
    {"stage": "smoothing",   "model": "Kalman (constant-vel)",   "params_M": 0.0, "coco_kp_mAP": None, "role": "per-track state smoothing + short-horizon predict"},
    {"stage": "motion",      "model": "MotionClassifier (rule)", "params_M": 0.0, "coco_kp_mAP": None, "role": "speed → still/walk/run hint for the predictor"},
    {"stage": "predict",     "model": "TrajectoryPredictor",     "params_M": 0.0, "coco_kp_mAP": None, "role": "k-step trajectory shaped by MotionHint"},
]).set_index("stage")
tracker

## 5. Face recognition (InsightFace `buffalo_l`)

`face_recognition_engine.py` loads the InsightFace `buffalo_l` bundle (ONNX) for SCRFD detection + ArcFace R100 embeddings, and caches enrolled embeddings to `embeddings_arcface.pkl`.

In [ ]:
face = pd.DataFrame([
    {"component": "SCRFD-10G (detector)",      "params_M": 3.86, "benchmark": "WIDERFACE Easy/Med/Hard", "score": "95.16 / 93.87 / 83.05 AP", "cpu_latency_ms": 5.0},
    {"component": "ArcFace R100 (embedding)",  "params_M": 65.1, "benchmark": "LFW / CFP-FP / AgeDB-30", "score": "99.83 / 98.85 / 98.27 acc",  "cpu_latency_ms": 12.0},
]).set_index("component")
face

## 6. SLAM — ORB keyframe VO + Depth Anything

`slam_engine.py` runs OpenCV ORB (1500 features, 8 levels) as the geometric front-end and uses the Depth Anything V2 variants above to lift 2D matches into a metric occupancy grid (vendored from MASt3R-SLAM's `OccupancyGrid2D`).

In [ ]:
slam = pd.DataFrame([
    {"component": "ORB detector/descriptor", "params_M": 0.0, "setting": "nfeatures=1500, scaleFactor=1.2, nlevels=8", "role": "keypoints + binary descriptors"},
    {"component": "KeyframeVO (PnP)",       "params_M": 0.0, "setting": "lowe_ratio=0.75, min_matches=25",          "role": "6-DoF pose relative to current keyframe"},
    {"component": "DepthAnythingV2Wrap",    "params_M": 97.5,"setting": "input_size=322, fp16",                     "role": "per-frame metric depth for unprojection"},
    {"component": "OccupancyGrid2D",        "params_M": 0.0, "setting": "MASt3R-SLAM vendor",                       "role": "2D top-down occupancy map"},
]).set_index("component")
slam

## 7. 3D reconstruction — LingBot-Map

`reconstruction_engine.py` shells out to an external LingBot-Map checkout (its own venv + `checkpoints/lingbot-map-long.pt`). Nothing about the network runs in-process, so the metrics below describe the external model only.

In [ ]:
recon = pd.DataFrame([
    {"model": "LingBot-Map (long)", "checkpoint": "lingbot-map-long.pt", "input": "monocular RGB stream", "output": "streaming dense 3D map", "runtime": "external subprocess"},
]).set_index("model")
recon

## 8. Cross-task summary

Side-by-side view of model size vs published per-frame latency. Useful for budgeting which engines can run concurrently on the same GPU during a flight.

*Placeholder values — see the inputs in the cell above.*

In [ ]:
summary = pd.DataFrame([
    {"engine": "RF-DETR Base",          "params_M": 29.0,  "latency_ms": 6.0},
    {"engine": "RF-DETR Medium",        "params_M": 38.0,  "latency_ms": 7.5},
    {"engine": "DA-V2 Indoor-Small",    "params_M": 24.8,  "latency_ms": 12.0},
    {"engine": "DA-V2 Indoor-Base",     "params_M": 97.5,  "latency_ms": 25.0},
    {"engine": "DA-V2 Outdoor-Small",   "params_M": 24.8,  "latency_ms": 12.0},
    {"engine": "DA-V2 Outdoor-Base",    "params_M": 97.5,  "latency_ms": 25.0},
    {"engine": "YOLOv8n-pose",          "params_M": 3.3,   "latency_ms": 4.0},
    {"engine": "DeepSort+MobileNet",    "params_M": 3.5,   "latency_ms": 3.0},
    {"engine": "SCRFD-10G",             "params_M": 3.86,  "latency_ms": 5.0},
    {"engine": "ArcFace R100",          "params_M": 65.1,  "latency_ms": 12.0},
]).set_index("engine")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.scatter(summary["params_M"], summary["latency_ms"], s=80, color="#4c78a8")
for name, row in summary.iterrows():
    ax.annotate(name, (row["params_M"], row["latency_ms"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("Params (M)")
ax.set_ylabel("Latency per frame (ms)")
ax.set_title("Walle GCS engines — size vs latency (placeholder values)")
plt.tight_layout(); plt.show()

## 9. Replacing the placeholders with real measurements

The metric columns above (`gpu_latency_ms`, `AbsRel`, `coco_mAP`, ...) are seeded with published numbers, not measurements from this hardware. To make this a real results page, swap each DataFrame with one of:

1. **Latency** — wrap the model call in `time.perf_counter()` inside `ObjectDetectionEngine._detection_worker`, `DepthAnythingV2Wrap.__call__`, `HumanTrackerEngine._run_loop`, and `FaceRecognitionEngine`. Log a moving average to CSV and load it here with `pd.read_csv`.
2. **Detection accuracy** — point RF-DETR at a labelled flight clip and compare boxes against ground truth (mAP via `pycocotools` or `supervision`).
3. **Depth error** — capture a few frames with a known scene scale (e.g. a 1 m ArUco marker on the floor) and compare DA-V2 metric depth against the marker distance.
4. **Tracking** — run on a short MOT-style clip and report IDF1 / MOTA via `motmetrics`.
5. **Face** — evaluate the cached `embeddings_arcface.pkl` against a held-out set of `known_faces/` images (cosine similarity threshold sweep).